In [1]:
import os 
import requests
import time
import pandas as pd 

In [2]:
  def fetch_nav(scheme_code):
    url = f'https://api.mfapi.in/mf/{scheme_code}'
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    result = response.json()
    meta = result['meta']
    df = pd.DataFrame(result['data'])
    return meta, df

In [3]:
meta, df = fetch_nav(119551)

print(meta['scheme_name'])
print(df.shape)
print(df.head())

Aditya Birla Sun Life Banking & PSU Debt Fund - Direct Plan - IDCW-Re-investment
(3326, 2)
         date        nav
0  09-10-2026  106.98300
1  08-10-2026  106.96630
2  07-10-2026  107.02850
3  06-10-2026  107.04410
4  05-10-2026  107.03330


In [4]:
schemes = {
    119551: 'SBI Bluechip',
    120503: 'ICICI Bluechip',
    118632: 'Nippon Large Cap',
    119092: 'Axis Bluechip',
    120841: 'Kotak Bluechip',
}

In [5]:
summary = []

for code, label in schemes.items():
    try:
        meta, df = fetch_nav(code)

        file_path = f'../data/raw/nav_{code}_raw.csv'
        df.to_csv(file_path, index=False)

        summary.append({
            'scheme_code': code,
            'task_name': label,
            'fund_house': meta['fund_house'],
            'api_name': meta['scheme_name'],
            'rows': len(df),
            'newest_date': df['date'].iloc[0],
            'oldest_date': df['date'].iloc[-1],
        })
        print('Saved', code)

    except requests.exceptions.RequestException as error:
        print('Failed', code, error)

    time.sleep(1)

Saved 119551
Saved 120503
Saved 118632
Saved 119092
Saved 120841


In [6]:
print(summary)

[{'scheme_code': 119551, 'task_name': 'SBI Bluechip', 'fund_house': 'Aditya Birla Sun Life Mutual Fund', 'api_name': 'Aditya Birla Sun Life Banking & PSU Debt Fund - Direct Plan - IDCW-Re-investment', 'rows': 3326, 'newest_date': '09-10-2026', 'oldest_date': '02-01-2013'}, {'scheme_code': 120503, 'task_name': 'ICICI Bluechip', 'fund_house': 'Axis Mutual Fund', 'api_name': 'Axis ELSS- Tax Saver Fund - Direct Plan - Growth Option', 'rows': 3398, 'newest_date': '09-10-2026', 'oldest_date': '02-01-2013'}, {'scheme_code': 118632, 'task_name': 'Nippon Large Cap', 'fund_house': 'Nippon India Mutual Fund', 'api_name': 'Nippon India Large Cap Fund - Direct Plan - Growth Option', 'rows': 3389, 'newest_date': '09-10-2026', 'oldest_date': '02-01-2013'}, {'scheme_code': 119092, 'task_name': 'Axis Bluechip', 'fund_house': 'HDFC Mutual Fund', 'api_name': 'HDFC Money Market Fund - Direct Plan - Growth Option', 'rows': 3655, 'newest_date': '09-10-2026', 'oldest_date': '31-12-2012'}, {'scheme_code': 120

In [7]:
pd.set_option('display.max_colwidth', None)

summary_df = pd.DataFrame(summary)
summary_df

,scheme_code,task_name,fund_house,api_name,rows,newest_date,oldest_date
0,119551,SBI Bluechip,Aditya Birla Sun Life Mutual Fund,Aditya Birla Sun Life Banking & PSU Debt Fund - Direct Plan - IDCW-Re-investment,3326,09-10-2026,02-01-2013
1,120503,ICICI Bluechip,Axis Mutual Fund,Axis ELSS- Tax Saver Fund - Direct Plan - Growth Option,3398,09-10-2026,02-01-2013
2,118632,Nippon Large Cap,Nippon India Mutual Fund,Nippon India Large Cap Fund - Direct Plan - Growth Option,3389,09-10-2026,02-01-2013
3,119092,Axis Bluechip,HDFC Mutual Fund,HDFC Money Market Fund - Direct Plan - Growth Option,3655,09-10-2026,31-12-2012
4,120841,Kotak Bluechip,quant Mutual Fund,Quant Mid Cap Fund - Direct Plan - Growth Option,3392,09-10-2026,07-01-2013


In [8]:
summary_df['is_direct'] = summary_df['api_name'].str.contains('Direct')
summary_df['is_growth'] = summary_df['api_name'].str.contains('Growth')

summary_df[['scheme_code', 'api_name', 'is_direct', 'is_growth']]

,scheme_code,api_name,is_direct,is_growth
0,119551,Aditya Birla Sun Life Banking & PSU Debt Fund - Direct Plan - IDCW-Re-investment,True,False
1,120503,Axis ELSS- Tax Saver Fund - Direct Plan - Growth Option,True,True
2,118632,Nippon India Large Cap Fund - Direct Plan - Growth Option,True,True
3,119092,HDFC Money Market Fund - Direct Plan - Growth Option,True,True
4,120841,Quant Mid Cap Fund - Direct Plan - Growth Option,True,True


In [9]:
sorted(os.listdir('../data/raw'))

['1788499980509-304c1255-08_investor_transactions.csv',
 '1788499982117-e3d6ab98-09_portfolio_holdings.csv',
 '1788499982615-f9647ab2-10_benchmark_indices.csv',
 '1788499983024-b042c300-01_fund_master.csv',
 '1788499983331-4389156d-02_nav_history.csv',
 '1788499984134-b0cbf625-03_aum_by_fund_house.csv',
 '1788499984405-d702a6c6-04_monthly_sip_inflows.csv',
 '1788499984721-4b860901-05_category_inflows.csv',
 '1788499985036-da4a0c4a-06_industry_folio_count.csv',
 '1788499985420-bb134abf-07_scheme_performance.csv',
 'nav_118632_raw.csv',
 'nav_119092_raw.csv',
 'nav_119551_raw.csv',
 'nav_120503_raw.csv',
 'nav_120841_raw.csv',
 'nav_125497_raw.csv']